# 02 — Бенчмарк SLAM на UCU SLAM Dataset v1
**Runtime → T4/L4/A100 GPU.** Потрібен секрет Colab `hf` (HF token із прийнятими умовами датасету) та готові збірки з `01_build_once.ipynb`.

Як це працює: 13 сіквенсів йдуть по черзі; наступний качається й розпаковується у фоні, поки рахується поточний. CPU-пайплайни
(ORB-SLAM3, DSO, OpenVINS…) і GPU-пайплайни (DPVO, DROID, SuperPoint+LightGlue+GTSAM, маски YOLO/MobileSAM) йдуть **паралельно**.
Після сіквенса сирі дані видаляються, результати лишаються на Drive: `slam_bench/results/seq-N/<pipeline>/`
(`traj_tum.txt`, `metrics.json`, `status.json`, `run.log`, `trajectory.png`, `trajectory.mp4`, native viewer відео де є).
Скрипт **відновлюється**: повторний запуск пропускає вже готове.

In [10]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)
HF_TOKEN = userdata.get('hf')

Mounted at /content/drive


In [13]:
import os
import subprocess

print("=== ПЕРЕВІРКА ТА ЗБІРКА БІНАРНИКІВ SLAM ===")

# 1. Перевірка та компіляція orb_run (ORB-SLAM3)
orb_bin = "/opt/slam/bin/orb_run"
if not os.path.exists(orb_bin):
    print("Компіляція orb_run...")
    orb_src = "/content/src/ORB_SLAM3"
    if os.path.exists(orb_src):
        # Якщо в джерелах є файл wrapper/main, збираємо його
        cmd = f"""
        cd {orb_src}
        mkdir -p build && cd build
        cmake .. -DCMAKE_INSTALL_PREFIX=/opt/slam
        make -j$(nproc)
        make install || cp -f orb_run /opt/slam/bin/ 2>/dev/null || true
        """
        subprocess.run(cmd, shell=True, check=False)

# 2. Перевірка та компіляція dso_dataset (DSO)
dso_bin = "/opt/slam/bin/dso_dataset"
if not os.path.exists(dso_bin):
    print("Компіляція dso_dataset...")
    dso_src = "/content/src/DSO"  # або папка з DSO у ваших джерелах
    if os.path.exists(dso_src):
        cmd = f"""
        cd {dso_src}
        mkdir -p build && cd build
        cmake .. -DCMAKE_INSTALL_PREFIX=/opt/slam
        make -j$(nproc)
        make install || cp -f bin/dso_dataset /opt/slam/bin/ 2>/dev/null || true
        """
        subprocess.run(cmd, shell=True, check=False)

# 3. Перевірка наявності бінарників
print("\n=== ВМІСТ /opt/slam/bin ===")
if os.path.exists("/opt/slam/bin"):
    print(os.listdir("/opt/slam/bin"))
else:
    print("Папка /opt/slam/bin відсутня")

=== ПЕРЕВІРКА ТА ЗБІРКА БІНАРНИКІВ SLAM ===
Компіляція orb_run...
Компіляція dso_dataset...

=== ВМІСТ /opt/slam/bin ===
['dso_run']


In [14]:
# %%bash
# set -e
# B=/content/drive/MyDrive/slam_bench/builds
# apt-get update -qq && apt-get install -y -qq build-essential cmake git pkg-config wget unzip libeigen3-dev libopencv-dev libboost-all-dev libssl-dev libglew-dev libepoxy-dev libegl1-mesa-dev libwayland-dev libxkbcommon-dev wayland-protocols libgl1-mesa-dev libsuitesparse-dev libzip-dev libgoogle-glog-dev libgflags-dev libatlas-base-dev libyaml-cpp-dev libceres-dev xvfb ffmpeg libjpeg-dev libpng-dev libtbb-dev > /dev/null
# tar xzf $B/slam_prefix.tar.gz -C /opt
# mkdir -p /content/src /content/work && tar xzf $B/sources.tar.gz -C /content/src
# pip install -q $B/wheels/*.whl
# pip install -q yacs einops numba plyfile pypose evo kornia tqdm scipy opencv-python pandas matplotlib \
#     ultralytics gtsam mcap mcap-ros2-support huggingface_hub
# pip install -q git+https://github.com/cvg/LightGlue.git
# TORCH=$(python -c "import torch;print(torch.__version__.split('+')[0])"); CU=cu$(python -c "import torch;print(torch.version.cuda.replace('.',''))")
# pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}+${CU}.html || pip install -q torch-scatter
# echo "build env was:"; head -3 $B/env.txt; echo "this env:"; python -c "import torch;print('torch',torch.__version__,'cuda',torch.version.cuda)"

In [11]:
from google.colab import drive
import os
import glob
import subprocess

# 1. Монтування Google Drive
print("=== 1. МОНТУВАННЯ GOOGLE DRIVE ===")
drive.mount('/content/drive', force_remount=True)

DRIVE_BUILDS = "/content/drive/MyDrive/slam_bench/builds"
WHEELS_DIR = os.path.join(DRIVE_BUILDS, "wheels")

# 2. Розпаковка системних префіксів C++ та джерельного коду
print("\n=== 2. РОЗПАКОВКА АРХІВІВ ===")
prefix_tar = os.path.join(DRIVE_BUILDS, "slam_prefix.tar.gz")
sources_tar = os.path.join(DRIVE_BUILDS, "sources.tar.gz")

if os.path.exists(prefix_tar):
    os.makedirs("/opt", exist_ok=True)
    subprocess.run(f"tar -xzf '{prefix_tar}' -C /opt", shell=True, check=True)
    print(" C++ prefix (/opt/slam) відновлено")

if os.path.exists(sources_tar):
    os.makedirs("/content/src", exist_ok=True)
    os.makedirs("/content/work", exist_ok=True)
    subprocess.run(f"tar -xzf '{sources_tar}' -C /content/src", shell=True, check=True)
    print(" Sources (/content/src) відновлено")

# 3. Встановлення зкомпільованих wheels
print("\n=== 3. ВСТАНОВЛЕННЯ WHEELS ===")
wheel_files = glob.glob(os.path.join(WHEELS_DIR, "*.whl"))
if not wheel_files:
    raise FileNotFoundError(f"Файли .whl не знайдено у {WHEELS_DIR}")

for whl in wheel_files:
    print(f"Встановлення {os.path.basename(whl)}...")
    subprocess.run(["pip", "install", whl, "--no-deps"], check=True)

# 4. Встановлення системних залежностей та Python-пакетів інфраструктури
print("\n=== 4. ВСТАНОВЛЕННЯ СУПУТНІХ БІБЛІОТЕК ===")
apt_cmd = (
    "apt-get update -qq && apt-get install -y -qq build-essential cmake git pkg-config "
    "wget unzip libeigen3-dev libopencv-dev libboost-all-dev libssl-dev libglew-dev libepoxy-dev "
    "libegl1-mesa-dev libwayland-dev libxkbcommon-dev wayland-protocols libgl1-mesa-dev "
    "libsuitesparse-dev libzip-dev libgoogle-glog-dev libgflags-dev libatlas-base-dev libyaml-cpp-dev "
    "libceres-dev xvfb ffmpeg libjpeg-dev libpng-dev libtbb-dev > /dev/null"
)
subprocess.run(apt_cmd, shell=True, check=True)

py_deps = [
    "yacs", "einops", "numba", "plyfile", "pypose", "evo", "kornia", "tqdm",
    "scipy", "opencv-python", "pandas", "matplotlib", "ultralytics", "gtsam",
    "mcap", "mcap-ros2-support", "huggingface_hub"
]
subprocess.run(["pip", "install", "-q"] + py_deps, check=True)
subprocess.run(["pip", "install", "-q", "git+https://github.com/cvg/LightGlue.git"], check=True)

# 5. Автоматична установка torch-scatter (з обробкою GPU / CPU)
import torch
torch_ver = torch.__version__.split('+')[0]
cuda_ver = torch.version.cuda

if cuda_ver:
    cu_str = f"cu{cuda_ver.replace('.', '')}"
    scatter_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cu_str}.html"
    subprocess.run(f"pip install -q torch-scatter -f {scatter_url} || pip install -q torch-scatter", shell=True)
else:
    subprocess.run(["pip", "install", "-q", "torch-scatter"])

# 6. Налаштування змінних оточення (PATH та LD_LIBRARY_PATH для Python 3.13)
os.environ["PATH"] = f"/opt/slam/bin:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/lib/python3.13/dist-packages/torch/lib:/opt/slam/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

print("\n=== 7. ФІНАЛЬНА ПЕРЕВІРКА ===")
subprocess.run(["python3", "-c", "import torch; import dpvo; print('DPVO OK')"], check=True)
subprocess.run(["python3", "-c", "import torch; import droid_backends; print('DROID-SLAM OK')"], check=True)

=== 1. МОНТУВАННЯ GOOGLE DRIVE ===
Mounted at /content/drive

=== 2. РОЗПАКОВКА АРХІВІВ ===
 C++ prefix (/opt/slam) відновлено
 Sources (/content/src) відновлено

=== 3. ВСТАНОВЛЕННЯ WHEELS ===
Встановлення dpvo-0.0.0-cp313-cp313-linux_x86_64.whl...
Встановлення droid_backends-0.0.0-cp313-cp313-linux_x86_64.whl...

=== 4. ВСТАНОВЛЕННЯ СУПУТНІХ БІБЛІОТЕК ===

=== 7. ФІНАЛЬНА ПЕРЕВІРКА ===


CompletedProcess(args=['python3', '-c', "import torch; import droid_backends; print('DROID-SLAM OK')"], returncode=0)

In [4]:
%%writefile /content/slam_common.py
"""Shared helpers: data download/extraction, evaluation, process runner, recording, tables."""
import os, sys, json, time, shutil, subprocess, signal, itertools, threading
from pathlib import Path
import numpy as np

DRIVE   = Path('/content/drive/MyDrive/slam_bench')
BUILDS  = DRIVE / 'builds'
RESULTS = DRIVE / 'results'
WORK    = Path('/content/work')
OPT     = Path('/opt/slam')            # unpacked C++ builds
SRC     = Path('/content/src')         # unpacked DPVO / DROID sources
REPO_ID = 'ucu-autonomous-ugv/ucu-slam-dataset-v1'


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


class NotAvailable(Exception):
    pass


# ------------------------------------------------------------------ data
def download_seq(n, token):
    os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(REPO_ID, f'sequence-{n}/sequence-{n}.mcap', repo_type='dataset',
                        token=token, local_dir=str(WORK / 'hf'))
    shutil.rmtree(Path.home() / '.cache/huggingface/xet', ignore_errors=True)
    return Path(p)


def drop_raw(n):
    shutil.rmtree(WORK / 'hf' / f'sequence-{n}', ignore_errors=True)
    shutil.rmtree(WORK / 'hf' / '.cache', ignore_errors=True)


def _stamp(h):
    return h.stamp.sec + h.stamp.nanosec * 1e-9


def _to_bgr(m):
    import cv2
    a = np.frombuffer(bytes(m.data), np.uint8)
    if m.encoding in ('rgb8', 'bgr8'):
        a = a.reshape(m.height, m.step)[:, :m.width * 3].reshape(m.height, m.width, 3)
        return np.ascontiguousarray(a[:, :, ::-1] if m.encoding == 'rgb8' else a)
    if m.encoding == 'mono8':
        a = a.reshape(m.height, m.step)[:, :m.width]
        return cv2.cvtColor(a, cv2.COLOR_GRAY2BGR)
    raise ValueError('unsupported image encoding ' + m.encoding)


def extract_seq(mcap_path, out):
    """MCAP -> rgb/*.jpg, rgb.txt, imu.txt, odom.txt, groundtruth.txt, calib.json, tf_static.json.
    Depth topics are NOT read (depth is broken in v1)."""
    import cv2
    from mcap_ros2.reader import read_ros2_messages
    out = Path(out)
    (out / 'rgb').mkdir(parents=True, exist_ok=True)
    rgb, imu, odom, gt, calib, tfs = [], [], [], [], {}, []
    topics = ['/cam_color/image_raw', '/cam_color/camera_info', '/imu', '/odom', '/pose', '/tf_static']
    for m in read_ros2_messages(str(mcap_path), topics=topics):
        tp, x = m.channel.topic, m.ros_msg
        if tp == '/cam_color/image_raw':
            t = _stamp(x.header)
            name = f'rgb/{t:017.6f}.jpg'
            cv2.imwrite(str(out / name), _to_bgr(x), [cv2.IMWRITE_JPEG_QUALITY, 95])
            rgb.append((t, name))
        elif tp == '/imu':
            a, w = x.linear_acceleration, x.angular_velocity
            imu.append([_stamp(x.header), w.x, w.y, w.z, a.x, a.y, a.z])
        elif tp in ('/odom', '/pose'):
            p, q = x.pose.pose.position, x.pose.pose.orientation
            (odom if tp == '/odom' else gt).append([_stamp(x.header), p.x, p.y, p.z, q.x, q.y, q.z, q.w])
        elif tp == '/cam_color/camera_info' and not calib:
            calib = dict(K=list(x.k), D=list(x.d), w=int(x.width), h=int(x.height), model=x.distortion_model)
        elif tp == '/tf_static':
            for tr in x.transforms:
                T, R = tr.transform.translation, tr.transform.rotation
                tfs.append(dict(parent=tr.header.frame_id, child=tr.child_frame_id,
                                t=[T.x, T.y, T.z], q=[R.x, R.y, R.z, R.w]))
    rgb.sort()
    with open(out / 'rgb.txt', 'w') as f:
        f.write('# color images\n# file: rgb.txt\n# timestamp filename\n')
        f.writelines(f'{t:.6f} {n}\n' for t, n in rgb)
    for name, arr in (('imu.txt', imu), ('odom.txt', odom), ('groundtruth.txt', gt)):
        arr = np.array(sorted(arr))
        np.savetxt(out / name, arr, fmt=['%.6f'] + ['%.9g'] * (arr.shape[1] - 1) if len(arr) else '%s')
    json.dump(calib, open(out / 'calib.json', 'w'), indent=1)
    json.dump(tfs, open(out / 'tf_static.json', 'w'), indent=1)
    log(f'extracted {out.name}: {len(rgb)} rgb, {len(imu)} imu, {len(odom)} odom, {len(gt)} gt')
    return len(rgb)


def tf_pose(tfs, target, source):
    """4x4 pose of frame `source` expressed in frame `target` (BFS over static tf tree)."""
    from scipy.spatial.transform import Rotation as Rot

    def mat(d):
        M = np.eye(4); M[:3, :3] = Rot.from_quat(d['q']).as_matrix(); M[:3, 3] = d['t']; return M
    edges = {}
    for d in tfs:
        M = mat(d)
        edges.setdefault(d['parent'], []).append((d['child'], M))
        edges.setdefault(d['child'], []).append((d['parent'], np.linalg.inv(M)))
    seen, queue = {target: np.eye(4)}, [target]
    while queue:
        a = queue.pop(0)
        for b, M in edges.get(a, []):
            if b not in seen:
                seen[b] = seen[a] @ M; queue.append(b)
    return seen.get(source)


# ------------------------------------------------------------------ evaluation
def load_tum(p):
    a = np.loadtxt(p, ndmin=2)
    return a[np.argsort(a[:, 0])]


def save_tum(p, arr):
    np.savetxt(p, arr, fmt=['%.6f'] + ['%.9g'] * 7)


def umeyama(X, Y, scale=True):
    """s,R,t minimising |Y - (s R X + t)|, X,Y: Nx3."""
    mx, my = X.mean(0), Y.mean(0)
    Xc, Yc = X - mx, Y - my
    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X))
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    s = (D * np.diag(S)).sum() / ((Xc ** 2).sum() / len(X)) if scale else 1.0
    return s, R, my - s * R @ mx


def aligned(est_file, gt_file, scale=True):
    est, gt = load_tum(est_file), load_tum(gt_file)
    m = (gt[:, 0] >= est[0, 0]) & (gt[:, 0] <= est[-1, 0])
    if m.sum() < 10:
        return None
    T, G = gt[m, 0], gt[m, 1:4]
    P = np.stack([np.interp(T, est[:, 0], est[:, i]) for i in (1, 2, 3)], 1)
    s, R, t = umeyama(P, G, scale)
    return T, (s * (R @ P.T)).T + t, G, s, est, gt


def evaluate(est_file, gt_file, radius=5.0, min_dt=30.0):
    """GT is position-only & meter-level, so: ATE on positions (Sim3 and SE3 alignment),
    horizontal ATE, and GT-revisit consistency as the loop-closure quality measure."""
    from scipy.spatial import cKDTree
    r = aligned(est_file, gt_file, True)
    if r is None:
        return dict(error='too few overlapping poses')
    T, A, G, s, est, gt = r
    out = dict(n_est=int(len(est)), n_gt=int(len(gt)), n_matched=int(len(T)),
               coverage=float((est[-1, 0] - est[0, 0]) / (gt[-1, 0] - gt[0, 0])), scale=float(s))
    for tag, sc in (('sim3', True), ('se3', False)):
        _, Aa, Gg, _, _, _ = aligned(est_file, gt_file, sc)
        d = Aa - Gg
        e, eh = np.linalg.norm(d, axis=1), np.linalg.norm(d[:, :2], axis=1)
        out.update({f'ate_{tag}_rmse': float(np.sqrt((e ** 2).mean())), f'ate_{tag}_mean': float(e.mean()),
                    f'ate_{tag}_median': float(np.median(e)), f'ate_{tag}_max': float(e.max()),
                    f'ate_{tag}_horiz_rmse': float(np.sqrt((eh ** 2).mean()))})
    # loop-closure quality: for GT revisits (close in space, far in time) how close are the *estimated* points
    pairs = cKDTree(G).query_pairs(radius, output_type='ndarray')
    if len(pairs):
        pairs = pairs[np.abs(T[pairs[:, 1]] - T[pairs[:, 0]]) > min_dt]
    out['loop_pairs'] = int(len(pairs))
    if len(pairs):
        err = np.linalg.norm(A[pairs[:, 0]] - A[pairs[:, 1]], axis=1)
        out.update(loop_revisit_mean=float(err.mean()), loop_revisit_median=float(np.median(err)),
                   loop_revisit_max=float(err.max()), loop_ok_ratio=float((err < 2 * radius).mean()))
    out['end_gap_est'] = float(np.linalg.norm(A[0] - A[-1]))
    out['end_gap_gt'] = float(np.linalg.norm(G[0] - G[-1]))
    return out


def plot_traj(est_file, gt_file, png, title=''):
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    r = aligned(est_file, gt_file, True)
    if r is None:
        return
    T, A, G, *_ = r
    plt.figure(figsize=(6, 6))
    plt.plot(G[:, 0], G[:, 1], 'k-', lw=1, label='GNSS GT')
    plt.plot(A[:, 0], A[:, 1], 'r-', lw=1, label='estimate (Sim3-aligned)')
    plt.axis('equal'); plt.legend(); plt.title(title); plt.grid(alpha=.3)
    plt.savefig(png, dpi=110, bbox_inches='tight'); plt.close()


def traj_video(est_file, gt_file, mp4, label='', n_frames=450, size=720):
    """Cheap, always-works animation of estimated vs GT trajectory."""
    import cv2
    r = aligned(est_file, gt_file, True)
    if r is None:
        return
    T, A, G, *_ = r
    allp = np.vstack([A[:, :2], G[:, :2]])
    lo, hi = allp.min(0), allp.max(0)
    sc = (size - 60) / max((hi - lo).max(), 1e-6)
    px = lambda P: ((P[:, :2] - lo) * sc + 30).astype(np.int32)
    pg, pa = px(G), px(A)
    vw = cv2.VideoWriter(str(mp4), cv2.VideoWriter_fourcc(*'mp4v'), 30, (size, size))
    for k in np.linspace(2, len(T) - 1, n_frames).astype(int):
        im = np.full((size, size, 3), 255, np.uint8)
        cv2.polylines(im, [pg.reshape(-1, 1, 2)], False, (150, 150, 150), 2)
        cv2.polylines(im, [pa[:k].reshape(-1, 1, 2)], False, (0, 0, 220), 2)
        cv2.circle(im, tuple(int(v) for v in pa[k]), 6, (0, 120, 0), -1)
        cv2.putText(im, f'{label}  t={T[k]-T[0]:.0f}s', (15, 25), cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 0, 0), 2)
        vw.write(im)
    vw.release()


# ------------------------------------------------------------------ processes
def run_cmd(cmd, logfile, timeout, cwd=None, env=None):
    e = os.environ.copy(); e.update({k: str(v) for k, v in (env or {}).items()})
    sh = isinstance(cmd, str)
    with open(logfile, 'w') as f:
        p = subprocess.Popen(cmd, shell=sh, executable='/bin/bash' if sh else None, stdout=f,
                             stderr=subprocess.STDOUT, cwd=cwd, env=e, start_new_session=True)
        try:
            return p.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL); p.wait()
            return 'TIMEOUT'


_disp = itertools.count(90)
_disp_lock = threading.Lock()


class X11Rec:
    """Virtual display + ffmpeg screen recording, for native viewers (Pangolin)."""
    def __init__(self, mp4, size='1280x720', fps=15):
        with _disp_lock:
            self.disp = f':{next(_disp)}'
        self.mp4, self.size, self.fps = str(mp4), size, fps
        self.env = {'DISPLAY': self.disp, 'LIBGL_ALWAYS_SOFTWARE': '1'}

    def __enter__(self):
        dn = subprocess.DEVNULL
        self.xvfb = subprocess.Popen(['Xvfb', self.disp, '-screen', '0', self.size + 'x24'], stdout=dn, stderr=dn)
        time.sleep(2)
        self.ff = subprocess.Popen(['ffmpeg', '-y', '-f', 'x11grab', '-framerate', str(self.fps), '-video_size',
                                    self.size, '-i', self.disp, '-c:v', 'libx264', '-preset', 'ultrafast',
                                    '-pix_fmt', 'yuv420p', self.mp4], stdin=subprocess.PIPE, stdout=dn, stderr=dn)
        return self

    def __exit__(self, *a):
        try:
            self.ff.stdin.write(b'q'); self.ff.stdin.flush(); self.ff.wait(30)
        except Exception:
            self.ff.kill()
        self.xvfb.terminate()


# ------------------------------------------------------------------ tables
def collect():
    import pandas as pd
    rows = []
    for st in RESULTS.glob('seq-*/*/status.json'):
        d = json.load(open(st))
        d['seq'] = int(st.parts[-3].split('-')[1]); d['pipeline'] = st.parts[-2]
        mf = st.parent / 'metrics.json'
        if mf.exists():
            d.update(json.load(open(mf)))
        rows.append(d)
    return pd.DataFrame(rows)


def show_tables(df=None, save=True):
    import pandas as pd
    from IPython.display import display
    df = collect() if df is None else df
    if df.empty:
        print('no results yet'); return df
    specs = [('ate_sim3_rmse', 'ATE RMSE, m (Sim3 align; scale-free)'), ('ate_sim3_mean', 'ATE mean, m (Sim3)'),
             ('ate_sim3_max', 'ATE MAX deviation, m (Sim3)'), ('ate_se3_rmse', 'ATE RMSE, m (SE3, metric scale kept)'),
             ('loop_revisit_mean', 'Loop closure: mean est. distance at GT revisits, m (lower=better)'),
             ('loop_ok_ratio', 'Loop closure: share of revisits closed (<10 m)'),
             ('end_gap_est', 'Start-end gap of estimate, m (GT gap in end_gap_gt)'),
             ('coverage', 'Trajectory coverage of GT time span'), ('runtime_s', 'Runtime, s'),
             ('status', 'Status')]
    out = RESULTS / '_tables'; out.mkdir(parents=True, exist_ok=True)
    pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
    for col, title in specs:
        if col not in df:
            continue
        pv = df.pivot(index='seq', columns='pipeline', values=col)
        if col != 'status':
            pv = pv.astype(float); pv.loc['MEAN'] = pv.mean(); pv.loc['MAX'] = pv.drop('MEAN').max()
            pv = pv.round(3)
        print('\n###', title); display(pv)
        if save:
            pv.to_csv(out / f'{col}.csv')
    if save:
        df.to_csv(out / 'all_metrics.csv', index=False)
    return df

Writing /content/slam_common.py


In [5]:
%%writefile /content/slam_pipelines.py
"""Pipeline wrappers. Each p_*(ctx) returns a path to a TUM trajectory: t x y z qx qy qz qw."""
import re, json, shutil, os
from pathlib import Path
import numpy as np
from slam_common import *

CFG = dict(
    stride_orb=2, stride_dso=2, stride_dpvo=2, stride_droid=6, stride_gtsam=6, stride_ov=2,
    orb_pace=2.0,                       # ORB-SLAM3 replay speed (x real time); too fast -> mapping can't keep up
    yolo_weights='yolov8n-seg.pt',
    dyn_classes=[0, 1, 2, 3, 5, 7, 14, 15, 16, 17],   # COCO: person, bicycle, car, motorcycle, bus, truck, bird, cat, dog, horse
    # !!! replace with the Allan-variance values from calibration.yaml (these are generic, inflated defaults)
    imu_noise=dict(gyro_nd=2.0e-3, gyro_rw=2.0e-5, acc_nd=2.0e-2, acc_rw=3.0e-3, rate=125.0),
    droid_buffer=1000,
)


class Ctx:
    def __init__(self, n, seq, calib, tfs, frames):
        self.n, self.seq, self.calib, self.tfs, self.frames = n, Path(seq), calib, tfs, frames
        self.res = RESULTS / f'seq-{n}'
        self.out = self.res; self.timeout = 900; self.viz = False; self.mask_ok = {}

    def child(self, name, timeout, viz):
        c = Ctx(self.n, self.seq, self.calib, self.tfs, self.frames)
        c.mask_ok = self.mask_ok; c.out = self.res / name; c.out.mkdir(parents=True, exist_ok=True)
        c.timeout, c.viz = timeout, viz
        return c


def load_ctx(n, seq):
    seq = Path(seq)
    frames = [(float(l.split()[0]), l.split()[1]) for l in open(seq / 'rgb.txt') if not l.startswith('#')]
    return Ctx(n, seq, json.load(open(seq / 'calib.json')), json.load(open(seq / 'tf_static.json')), frames)


# ------------------------------------------------------------------ input preparation
def prep_set(ctx, variant, stride):
    """Undistorted, strided image set for a variant (raw / yolo / sam) + all per-tool list files."""
    import cv2
    d = ctx.seq / f'set_{variant}_s{stride}'
    frames = ctx.frames[::stride]
    ts = np.array([t for t, _ in frames])
    K = np.array(ctx.calib['K']).reshape(3, 3); D = np.array(ctx.calib['D'][:5], float)
    W, H = ctx.calib['w'], ctx.calib['h']
    info = dict(dir=d, ts=ts, w=W, h=H, fx=K[0, 0], fy=K[1, 1], cx=K[0, 2], cy=K[1, 2])
    if (d / 'DONE').exists():
        return info
    (d / 'images').mkdir(parents=True, exist_ok=True)
    srcdir = ctx.seq / {'raw': 'rgb', 'yolo': 'rgb_yolo', 'sam': 'rgb_sam'}[variant]
    for i, (t, name) in enumerate(frames):
        p = srcdir / Path(name).name
        img = cv2.imread(str(p if p.exists() else ctx.seq / name))
        cv2.imwrite(str(d / 'images' / f'{i:06d}.jpg'), cv2.undistort(img, K, D), [cv2.IMWRITE_JPEG_QUALITY, 95])
    with open(d / 'rgb.txt', 'w') as f:
        f.write('# color images\n# file: rgb.txt\n# timestamp filename\n')
        f.writelines(f'{t:.6f} images/{i:06d}.jpg\n' for i, t in enumerate(ts))
    with open(d / 'times.txt', 'w') as f:
        f.writelines(f'{i} {t:.6f} 0.0\n' for i, t in enumerate(ts))
    open(d / 'camera.txt', 'w').write(f'Pinhole {info["fx"]} {info["fy"]} {info["cx"]} {info["cy"]} 0\n{W} {H}\ncrop\n{W} {H}\n')
    open(d / 'calib.txt', 'w').write(f'{info["fx"]} {info["fy"]} {info["cx"]} {info["cy"]}\n')
    (d / 'DONE').touch()
    return info


def remap_times(traj, ts):
    """Trajectories indexed by frame number -> real timestamps."""
    idx = traj[:, 0].astype(int)
    ok = (idx >= 0) & (idx < len(ts)) & np.allclose(traj[:, 0], idx)
    if not ok.all():                      # tool already wrote real timestamps
        return traj
    traj = traj.copy(); traj[:, 0] = ts[idx]
    return traj


def need(path, what):
    if not Path(path).exists():
        raise NotAvailable(f'{what} not found at {path} (run 01_build_once.ipynb)')


# ------------------------------------------------------------------ masks (YOLOv8-Seg / MobileSAM)
def make_masks(ctx, kind):
    """Blank out potentially dynamic objects (people, vehicles, animals). Every 2nd frame only (all strides nest)."""
    import cv2
    from ultralytics import YOLO, SAM
    out = ctx.seq / ('rgb_yolo' if kind == 'yolo' else 'rgb_sam'); out.mkdir(exist_ok=True)
    frames = ctx.frames[::2]
    yolo = YOLO(CFG['yolo_weights'])
    sam = SAM('mobile_sam.pt') if kind == 'sam' else None
    n_det, B = 0, 32
    for s in range(0, len(frames), B):
        chunk = frames[s:s + B]
        paths = [str(ctx.seq / 'rgb' / Path(n).name) for _, n in chunk]
        res = yolo.predict(paths, imgsz=640, conf=0.3, classes=CFG['dyn_classes'], retina_masks=True,
                           verbose=False, device=0)
        for (t, n), p, r in zip(chunk, paths, res):
            dst = out / Path(n).name
            if dst.exists():
                continue
            if r.boxes is None or len(r.boxes) == 0 or r.masks is None:
                os.symlink(p, dst); continue
            img = cv2.imread(p); H, W = img.shape[:2]
            if kind == 'yolo':
                m = r.masks.data.any(0).cpu().numpy().astype(np.uint8)
            else:
                rr = sam.predict(img, bboxes=r.boxes.xyxy.cpu().numpy().tolist(), verbose=False)
                m = rr[0].masks.data.any(0).cpu().numpy().astype(np.uint8)
            if m.shape != (H, W):
                m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
            img[cv2.dilate(m, np.ones((15, 15), np.uint8)) > 0] = 127
            cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95]); n_det += 1
    json.dump(dict(frames=len(frames), frames_with_masks=n_det), open(ctx.res / f'masks_{kind}.json', 'w'))
    log(f'seq {ctx.n}: {kind} masks on {n_det}/{len(frames)} frames')


# ------------------------------------------------------------------ ORB-SLAM3 (mono)
def orb_yaml(S, path, fps):
    open(path, 'w').write(f"""%YAML:1.0
Camera.type: "PinHole"
Camera.fx: {S['fx']}
Camera.fy: {S['fy']}
Camera.cx: {S['cx']}
Camera.cy: {S['cy']}
Camera.k1: 0.0
Camera.k2: 0.0
Camera.p1: 0.0
Camera.p2: 0.0
Camera.width: {S['w']}
Camera.height: {S['h']}
Camera.fps: {fps}
Camera.RGB: 0
ORBextractor.nFeatures: 1500
ORBextractor.scaleFactor: 1.2
ORBextractor.nLevels: 8
ORBextractor.iniThFAST: 20
ORBextractor.minThFAST: 7
Viewer.KeyFrameSize: 0.05
Viewer.KeyFrameLineWidth: 1.0
Viewer.GraphLineWidth: 0.9
Viewer.PointSize: 2.0
Viewer.CameraSize: 0.08
Viewer.CameraLineWidth: 3.0
Viewer.ViewpointX: 0.0
Viewer.ViewpointY: -0.7
Viewer.ViewpointZ: -1.8
Viewer.ViewpointF: 500.0
""")


def p_orb(ctx, variant='raw'):
    exe = OPT / 'bin/orb_run'; need(exe, 'orb_run'); need(OPT / 'share/ORBvoc.txt', 'ORB vocabulary')
    st = CFG['stride_orb']; S = prep_set(ctx, variant, st)
    yaml = ctx.out / 'orb_settings.yaml'; orb_yaml(S, yaml, 30 // st)
    cmd = [str(exe), str(OPT / 'share/ORBvoc.txt'), str(yaml), str(S['dir']), str(ctx.out),
           str(CFG['orb_pace']), '1' if ctx.viz else '0']
    env = {'LD_LIBRARY_PATH': f"{OPT}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')}
    if ctx.viz:
        with X11Rec(ctx.out / 'native_viewer.mp4') as rec:
            rc = run_cmd(cmd, ctx.out / 'run.log', ctx.timeout, env={**env, **rec.env})
    else:
        rc = run_cmd(cmd, ctx.out / 'run.log', ctx.timeout, env=env)
    txt = open(ctx.out / 'run.log').read()
    json.dump(dict(loops=len(re.findall(r'Loop detected', txt)), merges=len(re.findall(r'Merge detected', txt)),
                   rc=str(rc)), open(ctx.out / 'extra.json', 'w'))
    f = ctx.out / 'CameraTrajectory.txt'
    if rc != 0 and not f.exists():
        raise RuntimeError(f'orb_run rc={rc}, see run.log')
    return f


# ------------------------------------------------------------------ DSO
def p_dso(ctx, variant='raw'):
    exe = OPT / 'bin/dso_dataset'; need(exe, 'dso_dataset')
    S = prep_set(ctx, variant, CFG['stride_dso'])
    cmd = [str(exe), f"files={S['dir']}/images", f"calib={S['dir']}/camera.txt", 'mode=1', 'preset=0',
           'speed=0', 'quiet=1', 'nolog=1', f"nogui={0 if ctx.viz else 1}"]
    env = {'LD_LIBRARY_PATH': f"{OPT}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')}
    if ctx.viz:
        with X11Rec(ctx.out / 'native_viewer.mp4') as rec:
            rc = run_cmd(cmd, ctx.out / 'run.log', ctx.timeout, cwd=ctx.out, env={**env, **rec.env})
    else:
        rc = run_cmd(cmd, ctx.out / 'run.log', ctx.timeout, cwd=ctx.out, env=env)
    f = ctx.out / 'result.txt'
    if not f.exists():
        raise RuntimeError(f'dso rc={rc}, no result.txt, see run.log')
    tr = np.loadtxt(f, ndmin=2)[:, :8]
    # DSO writes the timestamp column from times.txt, but fall back to index mapping if needed
    save_tum(ctx.out / 'traj_tum.txt', remap_times(tr, S['ts']))
    return ctx.out / 'traj_tum.txt'


# ------------------------------------------------------------------ OpenVINS (standalone driver, no ROS)
def p_openvins(ctx):
    exe = OPT / 'bin/ov_run'; need(exe, 'ov_run')
    tpl = OPT / 'share/ov_template'; need(tpl / 'estimator_config.yaml', 'OpenVINS template config')
    Tic = tf_pose(ctx.tfs, 'imu', 'camera_color')          # pose of camera in IMU frame (cam -> imu)
    if Tic is None:
        raise NotAvailable('no static tf between imu and camera_color')
    c, K, D = ctx.calib, ctx.calib['K'], ctx.calib['D']
    cd = ctx.out / 'ov_cfg'; cd.mkdir(exist_ok=True)
    est = open(tpl / 'estimator_config.yaml').read()
    for k, v in (('max_cameras', '1'), ('use_stereo', 'false'), ('verbosity', '"WARNING"')):
        est = re.sub(rf'^{k}:.*$', f'{k}: {v}', est, flags=re.M)
    open(cd / 'estimator_config.yaml', 'w').write(est)
    n = CFG['imu_noise']
    imu = open(tpl / 'kalibr_imu_chain.yaml').read()
    for k, v in (('accelerometer_noise_density', n['acc_nd']), ('accelerometer_random_walk', n['acc_rw']),
                 ('gyroscope_noise_density', n['gyro_nd']), ('gyroscope_random_walk', n['gyro_rw']),
                 ('update_rate', n['rate'])):
        imu = re.sub(rf'^(\s*){k}:.*$', rf'\g<1>{k}: {v}', imu, flags=re.M)
    open(cd / 'kalibr_imu_chain.yaml', 'w').write(imu)
    rows = '\n'.join('    - [' + ', '.join(f'{x:.9f}' for x in r) + ']' for r in Tic)
    open(cd / 'kalibr_imucam_chain.yaml', 'w').write(f"""%YAML:1.0
cam0:
  T_imu_cam:
{rows}
  cam_overlaps: []
  camera_model: pinhole
  distortion_coeffs: [{', '.join(str(x) for x in D[:4])}]
  distortion_model: radtan
  intrinsics: [{K[0]}, {K[4]}, {K[2]}, {K[5]}]
  resolution: [{c['w']}, {c['h']}]
  rostopic: /cam0/image_raw
  timeshift_cam_imu: 0.0
""")
    out = ctx.out / 'traj_tum.txt'
    env = {'LD_LIBRARY_PATH': f"{OPT}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')}
    rc = run_cmd([str(exe), str(cd / 'estimator_config.yaml'), str(ctx.seq), str(out), str(CFG['stride_ov'])],
                 ctx.out / 'run.log', ctx.timeout, env=env)
    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError(f'ov_run rc={rc}, no output (init failed?), see run.log')
    return out


def p_dso_ov_fusion(ctx, win=30.0):
    """OUR INTERPRETATION of 'DSO + OpenVINS': loosely-coupled fusion. DSO gives the shape (no drift in scale
    within window), OpenVINS gives metric scale: DSO increments are rescaled per window so that the window path
    length matches OpenVINS. Output has positions only (identity orientation)."""
    a, b = ctx.res / 'DSO' / 'traj_tum.txt', ctx.res / 'OpenVINS' / 'traj_tum.txt'
    if not (a.exists() and b.exists()):
        raise NotAvailable('needs successful DSO and OpenVINS runs')
    d, o = load_tum(a), load_tum(b)
    t0, t1 = max(d[0, 0], o[0, 0]), min(d[-1, 0], o[-1, 0])
    d = d[(d[:, 0] >= t0) & (d[:, 0] <= t1)]
    if len(d) < 20:
        raise RuntimeError('no time overlap between DSO and OpenVINS')
    O = np.stack([np.interp(d[:, 0], o[:, 0], o[:, i]) for i in (1, 2, 3)], 1)
    s, R, t = umeyama(d[:, 1:4], O, True)                      # DSO -> OpenVINS frame
    P = (s * (R @ d[:, 1:4].T)).T + t
    out = np.zeros((len(d), 3)); out[0] = O[0]
    edges = np.arange(d[0, 0], d[-1, 0] + win, win)
    for lo, hi in zip(edges[:-1], edges[1:]):
        idx = np.where((d[:, 0] >= lo) & (d[:, 0] < hi))[0]
        if len(idx) < 2:
            continue
        lp = np.linalg.norm(np.diff(P[idx], axis=0), axis=1).sum()
        lo_ = np.linalg.norm(np.diff(O[idx], axis=0), axis=1).sum()
        k = np.clip(lo_ / lp, 0.5, 2.0) if lp > 1e-6 else 1.0
        for j in idx:
            if j > 0:
                out[j] = out[j - 1] + k * (P[j] - P[j - 1])
    res = np.zeros((len(d), 8)); res[:, 0] = d[:, 0]; res[:, 1:4] = out; res[:, 7] = 1
    save_tum(ctx.out / 'traj_tum.txt', res)
    return ctx.out / 'traj_tum.txt'


def p_vidso(ctx):
    raise NotAvailable('VI-DSO (Stumberg et al.) has no official public code. Provide a fork/binary and wire it here.')


# ------------------------------------------------------------------ DPVO
def p_dpvo(ctx, variant='raw'):
    src = SRC / 'DPVO'; need(src / 'demo.py', 'DPVO'); need(src / 'dpvo.pth', 'DPVO weights')
    S = prep_set(ctx, variant, CFG['stride_dpvo'])
    name = f'seq{ctx.n}_{variant}'
    base = f"python demo.py --imagedir {S['dir']}/images --calib {S['dir']}/calib.txt --stride 1 --name {name} " \
           f"--network dpvo.pth --config config/default.yaml --save_trajectory"
    f = src / 'saved_trajectories' / f'{name}.txt'
    f.unlink(missing_ok=True)
    rc = run_cmd(base + ' --opts BUFFER_SIZE 4096', ctx.out / 'run.log', ctx.timeout, cwd=src)
    if not f.exists() and rc != 'TIMEOUT':           # key may not exist in this DPVO version -> default config
        rc = run_cmd(base, ctx.out / 'run.log2', ctx.timeout, cwd=src)
    if not f.exists():
        raise RuntimeError(f'DPVO rc={rc}, no trajectory, see run.log')
    save_tum(ctx.out / 'traj_tum.txt', remap_times(np.loadtxt(f, ndmin=2)[:, :8], S['ts']))
    return ctx.out / 'traj_tum.txt'


# ------------------------------------------------------------------ DROID-SLAM
def p_droid(ctx, variant='raw'):
    src = SRC / 'DROID-SLAM'; need(src / 'droid_slam/droid.py', 'DROID-SLAM'); need(src / 'droid.pth', 'DROID weights')
    S = prep_set(ctx, variant, CFG['stride_droid'])
    npy = ctx.out / 'traj.npy'
    cmd = [sys.executable, str(OPT / 'share/run_droid.py'), '--src', str(src), '--imagedir', f"{S['dir']}/images",
           '--calib', f"{S['dir']}/calib.txt", '--out', str(npy), '--buffer', str(CFG['droid_buffer'])]
    rc = run_cmd(cmd, ctx.out / 'run.log', ctx.timeout)
    if not npy.exists():
        raise RuntimeError(f'DROID rc={rc}, no trajectory, see run.log')
    tr = np.load(npy)                                 # N x 7: t xyz qxyzw, camera-to-world
    arr = np.hstack([np.arange(len(tr))[:, None], tr])
    save_tum(ctx.out / 'traj_tum.txt', remap_times(arr, S['ts']))
    return ctx.out / 'traj_tum.txt'


# ------------------------------------------------------------------ SuperPoint + LightGlue -> GTSAM
def p_sp_lg_gtsam(ctx):
    """Own pipeline (depth is unusable in v1): SuperPoint+LightGlue -> essential-matrix relative poses (metric scale
    from wheel odometry) -> GTSAM iSAM2 pose graph; loop closures found by LightGlue verification of
    pose-proximal keyframes. GT is never used by the estimator."""
    import torch, cv2, gtsam
    from lightglue import LightGlue, SuperPoint
    from lightglue.utils import rbd
    S = prep_set(ctx, 'raw', CFG['stride_gtsam']); ts = S['ts']; N = len(ts)
    Kc = np.array([[S['fx'], 0, S['cx']], [0, S['fy'], S['cy']], [0, 0, 1.]])
    od = np.loadtxt(ctx.seq / 'odom.txt', ndmin=2)
    ox = np.stack([np.interp(ts, od[:, 0], od[:, i]) for i in (1, 2, 3)], 1)
    dev = 'cuda'
    ext = SuperPoint(max_num_keypoints=1024).eval().to(dev)
    mat = LightGlue(features='superpoint').eval().to(dev)

    def feats(i):
        im = cv2.cvtColor(cv2.imread(str(S['dir'] / 'images' / f'{i:06d}.jpg')), cv2.COLOR_BGR2RGB)
        with torch.no_grad():
            return ext.extract(torch.from_numpy(im).permute(2, 0, 1).float().div(255).to(dev))

    def rel(f0, f1):
        with torch.no_grad():
            m = mat({'image0': f0, 'image1': f1})
        a, b, m = rbd(f0), rbd(f1), rbd(m)
        mm = m['matches'].cpu().numpy()
        if len(mm) < 25:
            return None
        p0 = a['keypoints'][mm[:, 0]].cpu().numpy(); p1 = b['keypoints'][mm[:, 1]].cpu().numpy()
        E, mask = cv2.findEssentialMat(p0, p1, Kc, cv2.RANSAC, 0.999, 1.0)
        if E is None or E.shape != (3, 3):
            return None
        n, R, t, _ = cv2.recoverPose(E, p0, p1, Kc, mask=mask)
        return R, t.ravel(), int(n)

    def pose(R, t):    # cam1-from-cam0 [R|t]  ->  pose of cam1 in cam0 frame
        return gtsam.Pose3(gtsam.Rot3(R.T), gtsam.Point3(*(-R.T @ t)))

    isam = gtsam.ISAM2(gtsam.ISAM2Params())
    F = [feats(0)]
    g, v = gtsam.NonlinearFactorGraph(), gtsam.Values()
    g.add(gtsam.PriorFactorPose3(0, gtsam.Pose3(), gtsam.noiseModel.Diagonal.Sigmas(np.full(6, 1e-3))))
    v.insert(0, gtsam.Pose3()); isam.update(g, v)
    cur = [gtsam.Pose3()]; pos = [np.zeros(3)]; last_rel = gtsam.Pose3(); loops = []; cum = 0.0
    min_gap = max(40, int(20 * 30 / CFG['stride_gtsam']))
    for i in range(1, N):
        F.append(feats(i))
        d = float(np.linalg.norm(ox[i] - ox[i - 1])); cum += d
        r = rel(F[i - 1], F[i]) if d > 0.01 else None
        if r is not None and r[2] >= 30:
            rp = pose(r[0], r[1] / (np.linalg.norm(r[1]) + 1e-12) * d); sig = [0.03] * 3 + [0.05 * d + 0.01] * 3
            last_rel = rp
        else:                                      # stationary or visual failure -> hold last motion, wide noise
            rp = gtsam.Pose3(last_rel.rotation(), gtsam.Point3(*(last_rel.translation() / (np.linalg.norm(last_rel.translation()) + 1e-12) * d)))
            sig = [0.2] * 3 + [0.3 * d + 0.05] * 3
        g, v = gtsam.NonlinearFactorGraph(), gtsam.Values()
        g.add(gtsam.BetweenFactorPose3(i - 1, i, rp, gtsam.noiseModel.Diagonal.Sigmas(np.array(sig))))
        v.insert(i, cur[i - 1].compose(rp))
        # loop candidates: pose-proximal old keyframes, verified visually
        if i > min_gap:
            P = np.array(pos); dist = np.linalg.norm(P[:i - min_gap] - pos[i - 1], axis=1)
            radius = 8.0 + 0.03 * cum
            for j in np.argsort(dist)[:3]:
                if dist[j] > radius or any(abs(j - l[1]) < 15 and abs(i - l[0]) < 15 for l in loops):
                    continue
                r2 = rel(F[j], F[i])
                if r2 is None or r2[2] < 50:
                    continue
                dd = max(float(dist[j]), 1.0)
                lp = pose(r2[0], r2[1] / (np.linalg.norm(r2[1]) + 1e-12) * dd)
                huber = gtsam.noiseModel.Robust.Create(gtsam.noiseModel.mEstimator.Huber.Create(1.345),
                                                       gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1] * 3 + [0.5 * dd] * 3)))
                g.add(gtsam.BetweenFactorPose3(int(j), i, lp, huber)); loops.append((i, int(j), r2[2]))
        isam.update(g, v)
        if loops and loops[-1][0] == i:
            for _ in range(3):
                isam.update()
        est = isam.calculateEstimate()
        cur = [est.atPose3(k) for k in range(i + 1)] if (loops and loops[-1][0] == i) else cur + [est.atPose3(i)]
        pos = [p.translation() for p in cur]
        if i % 200 == 0:
            log(f'  gtsam kf {i}/{N} loops={len(loops)}')
    est = isam.calculateEstimate(); rows = []
    for k in range(N):
        p = est.atPose3(k); q = p.rotation().toQuaternion(); t = p.translation()
        rows.append([ts[k], t[0], t[1], t[2], q.x(), q.y(), q.z(), q.w()])
    save_tum(ctx.out / 'traj_tum.txt', np.array(rows))
    json.dump(dict(loops=len(loops), loop_edges=loops), open(ctx.out / 'extra.json', 'w'))
    return ctx.out / 'traj_tum.txt'

Writing /content/slam_pipelines.py


In [6]:
%%writefile /content/slam_run.py
"""Orchestrator: prefetch next sequence while the current one is processed; CPU and GPU pipelines run in parallel."""
import json, queue, shutil, threading, time, traceback
from concurrent.futures import ThreadPoolExecutor
from slam_common import *
from slam_pipelines import *


def safe(name):
    return re.sub(r'[^A-Za-z0-9_.+-]', '_', name)


def execute(ctx0, name, fn, timeout, viz=False, retry_failed=False):
    ctx = ctx0.child(safe(name), timeout, viz)
    st_file = ctx.out / 'status.json'
    if st_file.exists():
        old = json.load(open(st_file))['status']
        if old in ('OK', 'NOT_AVAILABLE') or (old == 'FAIL' and not retry_failed):
            return old
    t0, status, err = time.time(), 'OK', ''
    try:
        est = fn(ctx)
        if est is None or not Path(est).exists() or Path(est).stat().st_size == 0:
            raise RuntimeError('no trajectory produced')
        est = Path(est)
        if est.name != 'traj_tum.txt':
            shutil.copy(est, ctx.out / 'traj_tum.txt')
        gt = ctx.seq / 'groundtruth.txt'
        m = evaluate(ctx.out / 'traj_tum.txt', gt); m['runtime_s'] = round(time.time() - t0, 1)
        json.dump(m, open(ctx.out / 'metrics.json', 'w'), indent=1)
        if 'error' in m:
            status, err = 'FAIL', m['error']
        else:
            plot_traj(ctx.out / 'traj_tum.txt', gt, ctx.out / 'trajectory.png', f'seq {ctx.n} | {name}')
            traj_video(ctx.out / 'traj_tum.txt', gt, ctx.out / 'trajectory.mp4', f'seq {ctx.n} | {name}')
    except NotAvailable as e:
        status, err = 'NOT_AVAILABLE', str(e)
    except Exception:
        status, err = 'FAIL', traceback.format_exc()[-1500:]
    json.dump(dict(status=status, error=err, wall_s=round(time.time() - t0, 1)), open(st_file, 'w'), indent=1)
    log(f'seq {ctx.n:>2} | {name:<28} {status:<14} {time.time() - t0:6.0f}s {err.splitlines()[-1][:90] if err else ""}')
    return status


def _masked(ctx, kind, fn):
    def g(c):
        if not ctx.mask_ok.get(kind):
            raise RuntimeError(f'{kind} mask generation failed')
        return fn(c)
    return g


def run_sequence(n, seq_dir, budget, workers, viz, retry_failed):
    ctx = load_ctx(n, seq_dir); ctx.res.mkdir(parents=True, exist_ok=True)
    ev = {'yolo': threading.Event(), 'sam': threading.Event()}
    ex_ = lambda c, nm, f, to, v=False: execute(c, nm, f, to, v, retry_failed)

    cpu = [('ORB-SLAM3', lambda c: p_orb(c, 'raw'), None, viz),
           ('DSO', lambda c: p_dso(c, 'raw'), None, viz),
           ('OpenVINS', p_openvins, None, False),
           ('YOLOv8Seg+ORB-SLAM3', lambda c: p_orb(c, 'yolo'), 'yolo', False),
           ('YOLOv8Seg+DSO', lambda c: p_dso(c, 'yolo'), 'yolo', False),
           ('MobileSAM+ORB-SLAM3', lambda c: p_orb(c, 'sam'), 'sam', False)]
    gpu = [('SuperPoint+LightGlue+GTSAM', p_sp_lg_gtsam),
           ('DPVO', lambda c: p_dpvo(c, 'raw')),
           ('DROID-SLAM', lambda c: p_droid(c, 'raw')),
           ('YOLOv8Seg+DPVO', _masked(ctx, 'yolo', lambda c: p_dpvo(c, 'yolo'))),
           ('YOLOv8Seg+DROID-SLAM', _masked(ctx, 'yolo', lambda c: p_droid(c, 'yolo')))]
    t_cpu = max(150, 0.75 * budget * workers / len(cpu))
    t_gpu = max(150, 0.75 * budget / (len(gpu) + 1))

    def gpu_thread():
        for kind in ('yolo', 'sam'):
            try:
                make_masks(ctx, kind); ctx.mask_ok[kind] = True
            except Exception:
                log(f'seq {n}: mask {kind} FAILED\n' + traceback.format_exc()[-800:])
            ev[kind].set()
        for name, fn in gpu:
            ex_(ctx, name, fn, t_gpu)

    def cpu_task(name, fn, need_ev, v):
        if need_ev:
            ev[need_ev].wait()
            if not ctx.mask_ok.get(need_ev):
                fn = _masked(ctx, need_ev, fn)
        return ex_(ctx, name, fn, t_cpu, v)

    g = threading.Thread(target=gpu_thread); g.start()
    with ThreadPoolExecutor(workers) as pool:
        futs = [pool.submit(cpu_task, *t) for t in cpu]
        for f in futs:
            f.result()
    g.join()
    ex_(ctx, 'DSO+OpenVINS', p_dso_ov_fusion, 300)
    ex_(ctx, 'DSO+VI-DSO', p_vidso, 10)


def seq_done(n):
    names = ['ORB-SLAM3', 'DSO', 'OpenVINS', 'YOLOv8Seg+ORB-SLAM3', 'YOLOv8Seg+DSO', 'MobileSAM+ORB-SLAM3',
             'SuperPoint+LightGlue+GTSAM', 'DPVO', 'DROID-SLAM', 'YOLOv8Seg+DPVO', 'YOLOv8Seg+DROID-SLAM',
             'DSO+OpenVINS', 'DSO+VI-DSO']
    for nm in names:
        f = RESULTS / f'seq-{n}' / safe(nm) / 'status.json'
        if not f.exists():
            return False
    return True


def run_all(seqs, token, hours=6.0, workers=3, retry_failed=False, viz_seqs=(1,), keep_prefix=True):
    WORK.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
    deadline = time.time() + hours * 3600
    todo = [n for n in seqs if retry_failed or not seq_done(n)]
    log(f'sequences to process: {todo}')
    q = queue.Queue(maxsize=1)

    def producer():
        for n in todo:
            try:
                d = WORK / f'seq{n}'
                if not (d / 'rgb.txt').exists():
                    t0 = time.time(); raw = download_seq(n, token)
                    log(f'seq {n}: downloaded {raw.stat().st_size / 1e9:.1f} GB in {time.time() - t0:.0f}s')
                    shutil.rmtree(d, ignore_errors=True); extract_seq(raw, d); drop_raw(n)
                q.put((n, d))
            except Exception:
                log(f'seq {n}: download/extract FAILED\n' + traceback.format_exc()[-800:])
                drop_raw(n); q.put((n, None))
        q.put(None)

    threading.Thread(target=producer, daemon=True).start()
    left = len(todo)
    while True:
        item = q.get()
        if item is None:
            break
        n, d = item
        if time.time() > deadline:
            log('DEADLINE reached, stopping'); break
        if d is not None:
            budget = max(600, (deadline - time.time()) / left)
            log(f'=== seq {n} start (budget {budget / 60:.0f} min, {left} seq left) ===')
            try:
                run_sequence(n, d, budget, workers, n in viz_seqs, retry_failed)
            except Exception:
                log(f'seq {n}: run FAILED\n' + traceback.format_exc()[-1000:])
            shutil.rmtree(d, ignore_errors=True)          # "delete" the sequence, results stay on Drive
        left -= 1
    log('finished'); return collect()

Writing /content/slam_run.py


In [7]:
%%writefile /opt/slam/share/run_droid.py
"""Headless DROID-SLAM driver (based on the official demo.py). Saves camera-to-world poses (N x 7: xyz qxyzw)."""
import sys, os, argparse
p = argparse.ArgumentParser()
p.add_argument('--src'); p.add_argument('--imagedir'); p.add_argument('--calib'); p.add_argument('--out')
p.add_argument('--buffer', type=int, default=1000)
a = p.parse_args()
sys.path.insert(0, os.path.join(a.src, 'droid_slam'))
import numpy as np, cv2, torch
from droid import Droid


def image_stream():
    fx, fy, cx, cy = np.loadtxt(a.calib)[:4]
    for t, f in enumerate(sorted(os.listdir(a.imagedir))):
        im = cv2.imread(os.path.join(a.imagedir, f))
        h0, w0 = im.shape[:2]
        h1, w1 = int(h0 * np.sqrt((384 * 512) / (h0 * w0))), int(w0 * np.sqrt((384 * 512) / (h0 * w0)))
        im = cv2.resize(im, (w1, h1))[:h1 - h1 % 8, :w1 - w1 % 8]
        K = torch.as_tensor([fx, fy, cx, cy]); K[0::2] *= w1 / w0; K[1::2] *= h1 / h0
        yield t, torch.as_tensor(im).permute(2, 0, 1)[None], K


class Args:
    weights = os.path.join(a.src, 'droid.pth'); buffer = a.buffer; disable_vis = True; stereo = False
    beta = 0.3; filter_thresh = 2.4; warmup = 8; keyframe_thresh = 4.0; frontend_thresh = 16.0
    frontend_window = 25; frontend_radius = 2; frontend_nms = 1; backend_thresh = 22.0
    backend_radius = 2; backend_nms = 3; upsample = False; t0 = 0; image_size = None; reconstruction_path = None


args, droid = Args(), None
torch.multiprocessing.set_start_method('spawn', force=True)
for t, image, K in image_stream():
    if droid is None:
        args.image_size = [image.shape[2], image.shape[3]]
        droid = Droid(args)
    droid.track(t, image, intrinsics=K)
traj = droid.terminate(image_stream())
np.save(a.out, np.asarray(traj))

Overwriting /opt/slam/share/run_droid.py


In [24]:
import sys; sys.path.insert(0, '/content')
import importlib, slam_common, slam_pipelines, slam_run
for m in (slam_common, slam_pipelines, slam_run): importlib.reload(m)
from slam_common import *
from slam_pipelines import CFG

# --- IMU noise: take the real values from calibration.yaml (Allan variance) and put them here ---
import urllib.request
try:
    print(urllib.request.urlopen('https://raw.githubusercontent.com/ucu-autonomous-ugv/ucu-slam-dataset/main/calibration/calibration.yaml', timeout=20).read().decode()[:3000])
except Exception as e:
    print('could not fetch calibration.yaml:', e)
# CFG['imu_noise'] = dict(gyro_nd=..., gyro_rw=..., acc_nd=..., acc_rw=..., rate=125.0)

# --- quick availability check ---
chk = {'ORB-SLAM3': OPT/'bin/orb_run', 'DSO': OPT/'bin/dso_dataset', 'OpenVINS': OPT/'bin/ov_run',
       'DPVO': SRC/'DPVO/dpvo.pth', 'DROID': SRC/'DROID-SLAM/droid.pth'}
for k, v in chk.items(): print(f'{k:10s}', 'OK' if v.exists() else 'MISSING', v)

cameras:
  color:
    resolution: [640, 480] # [width, height] in [pixels]
    camera_model: pinhole
    distortion_model: radtan # radial-tangential
    intrinsics: [601.1733383650345, 600.2663647399592, 314.0164516341506, 250.039444021504] # [fx, fy, cx, cy] in [pixels]
    distortion_coeffs: [0.139268662386042, -0.30741279731381366, 0.0007272799549728076, 0.0006655761403425819] # [k1, k2, p1, p2]
    fps: 30

  depth:
    resolution: [640, 480]
    camera_model: pinhole
    distortion_model: none
    intrinsics: [384.009857178, 384.009857178, 321.232086182, 235.768280029]
    fps: 30
    depth_scale: 0.001 # [meters]

  depth_aligned:
    resolution: [640, 480]
    camera_model: pinhole
    distortion_model: radtan
    intrinsics: [601.1733383650345, 600.2663647399592, 314.0164516341506, 250.039444021504]
    distortion_coeffs: [0.139268662386042, -0.30741279731381366, 0.0007272799549728076, 0.0006655761403425819]
    fps: 30
    depth_scale: 0.001 # [meters]

transformations:
  # T

## Запуск
Спершу **смоук-тест на одному сіквенсі** (`SEQS=[1]`, `HOURS=0.6`) — це покаже, які пайплайни працюють на вашому образі Colab (статуси і хвости логів у `results/seq-1/*/run.log`).
Потім повний прогін. `HOURS` — загальна межа; бюджет на сіквенс = залишок часу / кількість сіквенсів, таймаути на пайплайни рахуються з нього.

In [29]:
import os
import subprocess

print("=== 1. НАДАННЯ ПРАВ НА ВИКОНАННЯ БІНАРНИКАМ ===")
# Виправляє PermissionError [Errno 13]
subprocess.run(["chmod", "-R", "+x", "/opt/slam/bin"], check=True)

print("=== 2. СТВОРЕННЯ СИМЛІНКІВ ТА ПЕРЕВІРКА КІНЦЕВИХ ПРАВ ===")
for b in ["orb_run", "dso_dataset", "ov_run"]:
    p = f"/opt/slam/bin/{b}"
    if os.path.exists(p):
        print(f" {b}: існує, права: {oct(os.stat(p).st_mode)[-3:]}")
    else:
        print(f" {b}: ВІДСУТНІЙ!")

=== 1. НАДАННЯ ПРАВ НА ВИКОНАННЯ БІНАРНИКАМ ===
=== 2. СТВОРЕННЯ СИМЛІНКІВ ТА ПЕРЕВІРКА КІНЦЕВИХ ПРАВ ===
 orb_run: існує, права: 711
 dso_dataset: існує, права: 711
 ov_run: існує, права: 711


In [ ]:
SEQS  = [12]                     # смоук-тест; для повного прогону: list(range(1, 14))
HOURS = 2.0                     # для повного прогону: 5.5
WORKERS = min(3, max(1, (os.cpu_count() or 2)))   # паралельних CPU-пайплайнів
VIZ_SEQS = (1,)                 # на яких сіквенсах писати відео нативних вʼюерів ORB-SLAM3/DSO (повільніше); (1,2,...,13) — на всіх
df = slam_run.run_all(SEQS, HF_TOKEN, hours=HOURS, workers=WORKERS, viz_seqs=VIZ_SEQS)

19:58:23 sequences to process: [12]


sequence-12/sequence-12.mcap: reconstructing file:   0%|          |  0.00B / 10.9GB            

sequence-12/sequence-12.mcap: downloading bytes:           |  0.00B            

20:03:48 seq 12: downloaded 10.9 GB in 325s
20:05:06 extracted seq12: 10908 rgb, 45543 imu, 6727 odom, 364 gt
20:05:07 === seq 12 start (budget 29 min, 1 seq left) ===
20:07:00 seq 12 | DSO                          FAIL              113s RuntimeError: dso rc=-11, no result.txt, see run.log
20:07:00 seq 12 | OpenVINS                     NOT_AVAILABLE       0s no static tf between imu and camera_color
20:07:01 seq 12 | ORB-SLAM3                    FAIL              114s RuntimeError: orb_run rc=1, see run.log
20:07:12 seq 12: yolo masks on 3597/5454 frames
20:08:49 seq 12 | YOLOv8Seg+DSO                FAIL               97s RuntimeError: dso rc=-11, no result.txt, see run.log
20:08:50 seq 12 | YOLOv8Seg+ORB-SLAM3          FAIL               98s RuntimeError: orb_run rc=1, see run.log
20:13:50 seq 12: sam masks on 3597/5454 frames
20:14:41   gtsam kf 200/1818 loops=0
20:14:55 seq 12 | MobileSAM+ORB-SLAM3          FAIL               65s RuntimeError: orb_run rc=1, see run.log
20:15:06   g

In [27]:
df = show_tables()


### ATE RMSE, m (Sim3 align; scale-free)


pipeline,DPVO,DROID-SLAM,DSO,DSO+OpenVINS,DSO+VI-DSO,MobileSAM+ORB-SLAM3,ORB-SLAM3,OpenVINS,SuperPoint+LightGlue+GTSAM,YOLOv8Seg+DPVO,YOLOv8Seg+DROID-SLAM,YOLOv8Seg+DSO,YOLOv8Seg+ORB-SLAM3
seq,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.425,NaN,NaN,NaN,NaN
MEAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.425,NaN,NaN,NaN,NaN
MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.425,NaN,NaN,NaN,NaN



### ATE mean, m (Sim3)


pipeline,DPVO,DROID-SLAM,DSO,DSO+OpenVINS,DSO+VI-DSO,MobileSAM+ORB-SLAM3,ORB-SLAM3,OpenVINS,SuperPoint+LightGlue+GTSAM,YOLOv8Seg+DPVO,YOLOv8Seg+DROID-SLAM,YOLOv8Seg+DSO,YOLOv8Seg+ORB-SLAM3
seq,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.299,NaN,NaN,NaN,NaN
MEAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.299,NaN,NaN,NaN,NaN
MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.299,NaN,NaN,NaN,NaN



### ATE MAX deviation, m (Sim3)


...

pipeline,DPVO,DROID-SLAM,DSO,DSO+OpenVINS,DSO+VI-DSO,MobileSAM+ORB-SLAM3,ORB-SLAM3,OpenVINS,SuperPoint+LightGlue+GTSAM,YOLOv8Seg+DPVO,YOLOv8Seg+DROID-SLAM,YOLOv8Seg+DSO,YOLOv8Seg+ORB-SLAM3
seq,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.004,NaN,NaN,NaN,NaN
MEAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.004,NaN,NaN,NaN,NaN
MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.004,NaN,NaN,NaN,NaN



### Runtime, s


pipeline,DPVO,DROID-SLAM,DSO,DSO+OpenVINS,DSO+VI-DSO,MobileSAM+ORB-SLAM3,ORB-SLAM3,OpenVINS,SuperPoint+LightGlue+GTSAM,YOLOv8Seg+DPVO,YOLOv8Seg+DROID-SLAM,YOLOv8Seg+DSO,YOLOv8Seg+ORB-SLAM3
seq,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.3,NaN,NaN,NaN,NaN
MEAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.3,NaN,NaN,NaN,NaN
MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.3,NaN,NaN,NaN,NaN



### Status


pipeline,DPVO,DROID-SLAM,DSO,DSO+OpenVINS,DSO+VI-DSO,MobileSAM+ORB-SLAM3,ORB-SLAM3,OpenVINS,SuperPoint+LightGlue+GTSAM,YOLOv8Seg+DPVO,YOLOv8Seg+DROID-SLAM,YOLOv8Seg+DSO,YOLOv8Seg+ORB-SLAM3
seq,,,,,,,,,,,,,
1,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,OK,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE,NOT_AVAILABLE


## Повторити те, що впало (після виправлення збірки/конфігу)
`retry_failed=True` перезапускає лише `FAIL`; `OK` і `NOT_AVAILABLE` не чіпає.

In [19]:
SEQS = list(range(1, 14)); HOURS = 2.0
df = slam_run.run_all(SEQS, HF_TOKEN, hours=HOURS, workers=WORKERS, viz_seqs=VIZ_SEQS, retry_failed=True)
show_tables()

19:14:53 sequences to process: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


sequence-1/sequence-1.mcap: reconstructing file:   0%|          |  0.00B / 6.51GB            

sequence-1/sequence-1.mcap: downloading bytes:           |  0.00B            

19:16:13 seq 1: downloaded 6.5 GB in 79s
19:16:58 extracted seq1: 6532 rgb, 27265 imu, 4026 odom, 218 gt
19:16:58 === seq 1 start (budget 10 min, 13 seq left) ===


sequence-2/sequence-2.mcap: reconstructing file:   0%|          |  0.00B / 10.2GB            

sequence-2/sequence-2.mcap: downloading bytes:           |  0.00B            

KeyboardInterrupt: 